# Stage 2 – Alzheimer MRI Image-Level Validation
**Project:** An Unsupervised Graph-Based Framework for Adversarial Attack Detection and Severity Estimation in Deep Learning-Based Alzheimer’s MRI Classification

This notebook validates the actual image files listed in the Stage 1 CSV. It checks file availability, image dimensions, colour modes, unreadable files, exact duplicates, and class-folder consistency.

**Important:** This stage does not train a model, resize images, augment data, or create train/test splits. We will decide those steps only after reviewing these validation results.

## 1. Upload the original dataset ZIP and Stage 1 CSV

In Google Colab, run the cell below and select:
- The original `Alzheimer (Preprocessed Data).zip` file
- Your `alzheimer_dataset_index.csv` file

If your Colab runtime still contains the extracted dataset, you may still upload the CSV; the notebook will reuse the extracted folder if found.

In [1]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Saving Alzheimer (Preprocessed Data) (1).zip to Alzheimer (Preprocessed Data) (1).zip
Uploaded files: ['Alzheimer (Preprocessed Data) (1).zip']


## 2. Locate the ZIP and CSV files

In [5]:
import os, glob, zipfile, hashlib, json, collections
import pandas as pd
from PIL import Image, ImageFile
from pathlib import Path

ImageFile.LOAD_TRUNCATED_IMAGES = False

csv_candidates = glob.glob("/content/**/*.csv", recursive=True)
zip_candidates = glob.glob("/content/**/*.zip", recursive=True)

print("CSV files found:")
for p in csv_candidates:
    print(" -", p)
print("\nZIP files found:")
for p in zip_candidates:
    print(" -", p)

CSV files found:
 - /content/sample_data/california_housing_test.csv
 - /content/sample_data/mnist_train_small.csv
 - /content/sample_data/california_housing_train.csv
 - /content/sample_data/mnist_test.csv

ZIP files found:
 - /content/Alzheimer (Preprocessed Data) (1).zip


If more than one CSV or ZIP appears, set the correct file paths explicitly in the next cell.

In [8]:
# Change these only if the automatic selection is not correct.
CSV_PATH = next((p for p in csv_candidates if os.path.basename(p) == "alzheimer_dataset_index.csv"), None)
ZIP_PATH = next((p for p in zip_candidates if "Alzheimer" in os.path.basename(p)), None)

print("CSV_PATH =", CSV_PATH)
print("ZIP_PATH =", ZIP_PATH)

if CSV_PATH is None:
    raise FileNotFoundError("Could not find alzheimer_dataset_index.csv. Upload it and rerun this cell.")

CSV_PATH = /content/alzheimer_dataset_index.csv
ZIP_PATH = /content/Alzheimer (Preprocessed Data) (1).zip


In [7]:
from google.colab import files

print("Please upload 'alzheimer_dataset_index.csv' below:")
uploaded_csv = files.upload()

# Verify and update the CSV path
import os, glob
csv_candidates = glob.glob("/content/**/*.csv", recursive=True)
CSV_PATH = next((p for p in csv_candidates if os.path.basename(p) == "alzheimer_dataset_index.csv"), None)
print("Updated CSV_PATH:", CSV_PATH)

Please upload 'alzheimer_dataset_index.csv' below:


Saving alzheimer_dataset_index.csv to alzheimer_dataset_index.csv
Updated CSV_PATH: /content/alzheimer_dataset_index.csv


## 3. Extract the dataset if needed

In [9]:
EXTRACT_DIR = "/content/alzheimer_mri_dataset"

# Reuse the Stage 1 extraction if the image directory is already present.
existing_images = glob.glob(EXTRACT_DIR + "/**/*.jpg", recursive=True)
if len(existing_images) == 0:
    if ZIP_PATH is None:
        raise FileNotFoundError("No extracted JPG images or dataset ZIP found. Upload the original ZIP.")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Dataset extracted.")
else:
    print("Existing extracted images found:", len(existing_images))

Dataset extracted.


## 4. Load and inspect the Stage 1 index

In [10]:
index_df = pd.read_csv(CSV_PATH)
required_cols = {"path", "label"}
if not required_cols.issubset(index_df.columns):
    raise ValueError(f"CSV must contain columns {required_cols}. Found: {index_df.columns.tolist()}")

print("Rows:", len(index_df))
print("Columns:", index_df.columns.tolist())
print("\nClass counts:")
display(index_df["label"].value_counts().rename_axis("Class").to_frame("Images"))
print("\nSample paths:")
display(index_df.head())

Rows: 6400
Columns: ['path', 'label']

Class counts:


,Images
Class,
Non-Demented,3200
Very Mild Demented,2240
Mild Demented,896
Moderate Demented,64



Sample paths:


,path,label
0,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Moderate Demented
1,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Moderate Demented
2,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Moderate Demented
3,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Moderate Demented
4,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Moderate Demented


## 5. Resolve each CSV path to an image file

The Stage 1 CSV contains Colab paths beginning with `/content/alzheimer_mri_dataset`. This cell first checks those exact paths, then tries to locate the image by its relative path inside the extracted dataset.

In [11]:
def resolve_image_path(csv_path):
    if os.path.isfile(csv_path):
        return csv_path
    marker = "Alzheimer (Preprocessed Data)/"
    if marker in csv_path:
        relative = csv_path.split(marker, 1)[1]
        matches = glob.glob(EXTRACT_DIR + "/**/" + os.path.basename(relative), recursive=True)
        # Prefer a match whose trailing folder structure matches the CSV path.
        for candidate in matches:
            if candidate.replace("\\", "/").endswith(relative.replace("\\", "/")):
                return candidate
        if len(matches) == 1:
            return matches[0]
    basename_matches = glob.glob(EXTRACT_DIR + "/**/" + os.path.basename(csv_path), recursive=True)
    return basename_matches[0] if len(basename_matches) == 1 else None

index_df["resolved_path"] = index_df["path"].apply(resolve_image_path)
missing_files = index_df[index_df["resolved_path"].isna()].copy()
print("Resolved files:", index_df["resolved_path"].notna().sum())
print("Missing/unresolved files:", len(missing_files))
if len(missing_files):
    display(missing_files.head(20))

Resolved files: 6400
Missing/unresolved files: 0


## 6. Inspect image dimensions, colour modes and file readability

In [12]:
records = []
unreadable = []

for row in index_df.itertuples(index=False):
    p = row.resolved_path
    if not p:
        continue
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            records.append({
                "path": row.path,
                "resolved_path": p,
                "label": row.label,
                "width": im.width,
                "height": im.height,
                "mode": im.mode,
                "format": im.format,
                "file_bytes": os.path.getsize(p)
            })
    except Exception as e:
        unreadable.append({"path": row.path, "label": row.label, "error": str(e)})

image_report = pd.DataFrame(records)
unreadable_df = pd.DataFrame(unreadable)

print("Readable images:", len(image_report))
print("Unreadable images:", len(unreadable_df))
print("\nImage dimensions:")
display(image_report.groupby(["width", "height"]).size().reset_index(name="count"))
print("\nColour modes:")
display(image_report["mode"].value_counts().rename_axis("Mode").to_frame("Count"))
print("\nFormats:")
display(image_report["format"].value_counts().rename_axis("Format").to_frame("Count"))

Readable images: 6400
Unreadable images: 0

Image dimensions:


,width,height,count
0,128,128,6400



Colour modes:


,Count
Mode,
L,6400



Formats:


,Count
Format,
JPEG,6400


## 7. Check exact duplicate files using SHA-256

In [13]:
hash_records = []
for row in image_report.itertuples(index=False):
    h = hashlib.sha256()
    with open(row.resolved_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    hash_records.append({
        "path": row.path,
        "resolved_path": row.resolved_path,
        "label": row.label,
        "sha256": h.hexdigest()
    })

hash_df = pd.DataFrame(hash_records)
duplicate_hashes = hash_df[hash_df.duplicated("sha256", keep=False)].sort_values("sha256")
print("Number of files participating in exact-duplicate groups:", len(duplicate_hashes))
print("Number of distinct duplicate hashes:", duplicate_hashes["sha256"].nunique() if len(duplicate_hashes) else 0)
if len(duplicate_hashes):
    display(duplicate_hashes.head(30))

Number of files participating in exact-duplicate groups: 0
Number of distinct duplicate hashes: 0


## 8. Check whether the folder name agrees with the CSV label

In [14]:
def folder_label(path):
    parent = os.path.basename(os.path.dirname(path)).replace("_", " ").strip().lower()
    return parent

label_check = image_report.copy()
label_check["folder_label"] = label_check["resolved_path"].apply(folder_label)
label_check["label_normalized"] = label_check["label"].str.strip().str.lower()
label_check["folder_matches_label"] = label_check["folder_label"] == label_check["label_normalized"]

print("Folder/label mismatches:", int((~label_check["folder_matches_label"]).sum()))
if (~label_check["folder_matches_label"]).any():
    display(label_check.loc[~label_check["folder_matches_label"], ["resolved_path", "label", "folder_label"]].head(30))

Folder/label mismatches: 3200


,resolved_path,label,folder_label
64,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
65,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
66,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
67,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
68,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
69,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
70,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
71,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
72,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented
73,/content/alzheimer_mri_dataset/Alzheimer (Prep...,Non-Demented,non demented


## 9. Summarize validation by class

In [15]:
summary = image_report.groupby("label").agg(
    readable_images=("resolved_path", "count"),
    unique_dimensions=("width", lambda s: "see dimension table"),
    modes=("mode", lambda s: ", ".join(sorted(set(s))))
).reset_index()

print("CSV class counts:")
display(index_df["label"].value_counts().rename_axis("label").to_frame("CSV_rows"))
print("Readable image counts:")
display(image_report["label"].value_counts().rename_axis("label").to_frame("Readable_images"))
print("Validation summary:")
display(summary)

CSV class counts:


,CSV_rows
label,
Non-Demented,3200
Very Mild Demented,2240
Mild Demented,896
Moderate Demented,64


Readable image counts:


,Readable_images
label,
Non-Demented,3200
Very Mild Demented,2240
Mild Demented,896
Moderate Demented,64


Validation summary:


,label,readable_images,unique_dimensions,modes
0,Mild Demented,896,see dimension table,L
1,Moderate Demented,64,see dimension table,L
2,Non-Demented,3200,see dimension table,L
3,Very Mild Demented,2240,see dimension table,L


## 10. Save the Stage 2 validation report

In [16]:
os.makedirs("/content/stage2_validation", exist_ok=True)

index_df.to_csv("/content/stage2_validation/stage1_index_with_resolved_paths.csv", index=False)
image_report.to_csv("/content/stage2_validation/image_file_report.csv", index=False)
hash_df.to_csv("/content/stage2_validation/sha256_hashes.csv", index=False)
duplicate_hashes.to_csv("/content/stage2_validation/exact_duplicates.csv", index=False)
unreadable_df.to_csv("/content/stage2_validation/unreadable_images.csv", index=False)
label_check.to_csv("/content/stage2_validation/folder_label_check.csv", index=False)

validation_summary = {
    "csv_rows": int(len(index_df)),
    "resolved_files": int(index_df["resolved_path"].notna().sum()),
    "unresolved_files": int(index_df["resolved_path"].isna().sum()),
    "readable_images": int(len(image_report)),
    "unreadable_images": int(len(unreadable_df)),
    "exact_duplicate_file_rows": int(len(duplicate_hashes)),
    "folder_label_mismatches": int((~label_check["folder_matches_label"]).sum()) if len(label_check) else None,
    "dimensions": image_report.groupby(["width", "height"]).size().to_dict().__str__(),
    "colour_modes": image_report["mode"].value_counts().to_dict()
}
with open("/content/stage2_validation/validation_summary.json", "w") as f:
    json.dump(validation_summary, f, indent=2)

print(json.dumps(validation_summary, indent=2))

{
  "csv_rows": 6400,
  "resolved_files": 6400,
  "unresolved_files": 0,
  "readable_images": 6400,
  "unreadable_images": 0,
  "exact_duplicate_file_rows": 0,
  "folder_label_mismatches": 3200,
  "dimensions": "{(128, 128): 6400}",
  "colour_modes": {
    "L": 6400
  }
}


## 11. Download the validation report

In [17]:
import shutil
shutil.make_archive("/content/Stage2_Validation_Report", "zip", "/content/stage2_validation")
files.download("/content/Stage2_Validation_Report.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What to send back

Upload `Stage2_Validation_Report.zip` here, or paste:
1. Readable and unreadable image counts
2. Image dimension table
3. Colour mode counts
4. Exact duplicate counts
5. Folder/label mismatch count

**Do not create a train/validation/test split yet.** After reviewing these results, we will choose and document a reproducible split strategy. The available CSV has no patient identifiers, so we must state that limitation accurately and avoid claiming patient-level separation.